[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/02_%E7%B5%B1%E8%A8%88_4%E7%B4%9A/03_%E3%81%B0%E3%82%89%E3%81%A4%E3%81%8D%E3%81%A8%E7%AE%B1%E3%81%B2%E3%81%92%E5%9B%B3.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計4級-03. ばらつき・四分位数・箱ひげ図

平均が同じでも「散らばり方」は違います。それを表す方法を学びます。

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画

# 日本語が文字化けするときは次の1行を有効にしてください
# plt.rcParams['font.family'] = 'Hiragino Sans'  # Macの場合
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
df = pd.read_csv('../data/students_scores.csv')   # 生徒データを読み込む
df.head()                        # 先頭5行を確認

### 📋 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

## 1. 範囲（レンジ）

いちばん簡単なばらつきの指標。`最大値 − 最小値`。

In [ ]:
math = df['数学']                          # 数学の列
print('最大:', math.max(), '最小:', math.min())  # 最大値と最小値
print('範囲:', math.max() - math.min())    # 範囲 ＝ 最大 − 最小

## 2. 四分位数

データを小さい順に並べ、4等分する境目の値です。

- **第1四分位数 (Q1)**：下から25%の位置
- **第2四分位数 (Q2)**：中央値（50%）
- **第3四分位数 (Q3)**：下から75%の位置
- **四分位範囲 (IQR)** = Q3 − Q1 … 真ん中50%の散らばり

In [ ]:
q1 = math.quantile(0.25)   # 第1四分位数（下から25%の位置）
q2 = math.quantile(0.50)   # 第2四分位数（＝中央値）
q3 = math.quantile(0.75)   # 第3四分位数（下から75%の位置）
print('Q1:', q1, ' Q2(中央値):', q2, ' Q3:', q3)
print('IQR:', q3 - q1)     # 四分位範囲 ＝ Q3 − Q1

# describe() で一気に確認できる
math.describe()            # 件数・平均・四分位などをまとめて表示

## 3. 箱ひげ図

四分位数を図にしたもの。複数グループの比較に最適です。

In [ ]:
plt.figure(figsize=(7, 4))                 # グラフの大きさ
df.boxplot(column='数学', by='クラス')      # クラス別に箱ひげ図
plt.title('クラス別 数学の点数')            # タイトル
plt.suptitle('')                           # 自動で付く上の見出しを消す
plt.ylabel('点数')                         # y軸ラベル
plt.show()

**箱ひげ図の読み方**
- 箱の下辺=Q1、中の線=中央値、上辺=Q3
- ひげの先=外れ値を除いた最大・最小
- 箱が長い＝ばらつきが大きい

## 4. 外れ値の見つけ方

ふつう `Q1 − 1.5×IQR` より小さい、または `Q3 + 1.5×IQR` より大きい値を外れ値とします。

In [ ]:
iqr = q3 - q1                              # 四分位範囲
low = q1 - 1.5 * iqr                       # 外れ値とみなす下限
high = q3 + 1.5 * iqr                      # 外れ値とみなす上限
print(f'外れ値の境界: {low:.1f} 未満 または {high:.1f} 超')
df[(df['数学'] < low) | (df['数学'] > high)][['生徒ID', '数学']]  # 範囲外の生徒を抽出

---
## 🏆 練習問題

**問1.** `[12, 15, 18, 22, 25, 30, 35]` の Q1・Q2・Q3・IQR を求めよう。

**問2.** `国語` の箱ひげ図をクラス別に描き、最もばらつきが大きいクラスを答えよう。

**問3.** `英語` で外れ値があるか、IQRの方法で調べよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


<details>
<summary>▶ 解答例を見る（クリックで開く）</summary>

<pre style="background:#f6f8fa;padding:10px;border-radius:6px;overflow:auto"><code>x = pd.Series([12,15,18,22,25,30,35])
print(x.quantile([.25,.5,.75]))
df.boxplot(column=&#x27;国語&#x27;, by=&#x27;クラス&#x27;); plt.show()</code></pre>
</details>